<a href="https://colab.research.google.com/github/nguyenkyvi/Machine_Learning/blob/main/Lab05_Decision%20Tree_Random%20Forest/Lab05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text

# Mười khách vay để tính tay, giống hệt bảng trên slide[cite: 27]
nho = pd.DataFrame({
    "thu_nhap": [6, 8, 9, 11, 13, 15, 18, 20, 25, 30],
    "co_viec_on_dinh": [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
    "tra_dung_han": [0, 1, 0, 0, 1, 1, 0, 1, 1, 1],
})
y = nho["tra_dung_han"].to_numpy()


def gini(nhan):
    """Chi so Gini cua mot nut co hai lop."""
    if len(nhan) == 0:
        return 0.0
    p1 = nhan.mean()
    return 1 - p1 ** 2 - (1 - p1) ** 2


def gini_sau_chia(di_trai):
    """Gini trung binh co trong so cua hai nhanh."""
    trai, phai = y[di_trai], y[~di_trai]
    n = len(y)
    return len(trai) / n * gini(trai) + len(phai) / n * gini(phai)


g_goc = gini(y)
print(f"Gini cua nut goc: {g_goc:.4f}")
print()

# Thử mọi ngưỡng nằm giữa hai mức thu nhập liền nhau[cite: 28]
thu = nho["thu_nhap"].to_numpy()
muc = np.sort(np.unique(thu))
print("Cau hoi 'thu_nhap <= nguong ?'")
print(" nguong   Gini sau chia   Gini giam")
for a, b in zip(muc[:-1], muc[1:]):
    nguong = (a + b) / 2
    g = gini_sau_chia(thu <= nguong)
    print(f" {nguong:6.1f}   {g:13.4f}   {g_goc - g:9.4f}")

print()

# Câu hỏi thứ hai: có việc làm ổn định không[cite: 28]
viec = nho["co_viec_on_dinh"].to_numpy()
g_viec = gini_sau_chia(viec == 0)
print("Cau hoi 'co viec on dinh ?'")
print(f" Gini sau chia {g_viec:.4f}, Gini giam {g_goc - g_viec:.4f}")
print()

# Để scikit-learn tự chọn câu hỏi đầu tiên rồi so với bảng ở trên[cite: 28]
cot = ["thu_nhap", "co_viec_on_dinh"]
cay = DecisionTreeClassifier(max_depth=1, random_state=0)
cay.fit(nho[cot], y)
print("Cay mot tang do scikit-learn chon:")
print(export_text(cay, feature_names=cot, show_weights=True))

Gini cua nut goc: 0.4800

Cau hoi 'thu_nhap <= nguong ?'
 nguong   Gini sau chia   Gini giam
    7.0          0.4000      0.0800
    8.5          0.4750      0.0050
   10.0          0.4190      0.0610
   12.0          0.3167      0.1633
   14.0          0.4000      0.0800
   16.5          0.4500      0.0300
   19.0          0.3429      0.1371
   22.5          0.4000      0.0800
   27.5          0.4444      0.0356

Cau hoi 'co viec on dinh ?'
 Gini sau chia 0.4190, Gini giam 0.0610

Cay mot tang do scikit-learn chon:
|--- thu_nhap <= 12.00
|   |--- weights: [3.00, 1.00] class: 0
|--- thu_nhap >  12.00
|   |--- weights: [1.00, 5.00] class: 1



In [5]:
# -*- coding: utf-8 -*-
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

df = pd.read_csv("data/khach_vay.csv")
print("Kich thuoc bang:", df.shape)
print(df.head())
print()
print("Ty le khach tra dung han:", round(df["tra_dung_han"].mean(), 2))
print()

X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Giới hạn cây chỉ được có 5 lá để còn đọc được bằng mắt
cay = DecisionTreeClassifier(max_leaf_nodes=5, random_state=0)
cay.fit(X_tr, y_tr)

print(f"Do sau {cay.get_depth()}, so la {cay.get_n_leaves()}")
print(
    export_text(
        cay, feature_names=list(X.columns), show_weights=True, decimals=3
    )
)

a_hoc = accuracy_score(y_tr, cay.predict(X_tr))
a_kt = accuracy_score(y_te, cay.predict(X_te))
print(f"Accuracy tap hoc     : {a_hoc:.4f}")
print(f"Accuracy tap kiem tra: {a_kt:.4f}")
print()

# Hai khách mới, đi theo cây để biết mỗi người rơi vào lá nào
moi = pd.DataFrame({
    "thu_nhap": [20.0, 10.0],
    "ty_le_no": [0.4, 0.3],
    "co_viec_on_dinh": [1, 0],
    "so_nguoi_phu_thuoc": [2, 1],
})
du_doan = cay.predict(moi)
xac_suat = cay.predict_proba(moi)[:, 1]
for i in range(len(moi)):
  ket = "dung han" if du_doan[i] == 1 else "tre han"
  print(f"Khach {i + 1}: {ket}, xac suat dung han {xac_suat[i]:.2f}")

Kich thuoc bang: (600, 5)
   thu_nhap  ty_le_no  co_viec_on_dinh  so_nguoi_phu_thuoc  tra_dung_han
0       9.4      0.18                0                   4             0
1      15.8      0.70                0                   2             1
2       6.1      0.79                0                   0             0
3      19.4      1.13                1                   4             0
4      24.8      0.33                1                   0             1

Ty le khach tra dung han: 0.49

Do sau 3, so la 5
|--- thu_nhap <= 15.050
|   |--- ty_le_no <= 0.615
|   |   |--- co_viec_on_dinh <= 0.500
|   |   |   |--- weights: [33.000, 6.000] class: 0
|   |   |--- co_viec_on_dinh >  0.500
|   |   |   |--- weights: [35.000, 47.000] class: 1
|   |--- ty_le_no >  0.615
|   |   |--- weights: [114.000, 14.000] class: 0
|--- thu_nhap >  15.050
|   |--- ty_le_no <= 1.020
|   |   |--- weights: [27.000, 145.000] class: 1
|   |--- ty_le_no >  1.020
|   |   |--- weights: [20.000, 9.000] class: 0

Accu

In [6]:
# -*- coding: utf-8 -*-
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("data/khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Không đặt giới hạn nào, cây chia tới khi mọi lá đều thuần[cite: 36]
tu_do = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
print("Cay moc tu do")
print(f" do sau {tu_do.get_depth()}, so la {tu_do.get_n_leaves()}")
a_hoc = accuracy_score(y_tr, tu_do.predict(X_tr))
a_kt = accuracy_score(y_te, tu_do.predict(X_te))
print(f" tap hoc {a_hoc:.4f}, tap kiem tra {a_kt:.4f}")
print()

# Chọn độ sâu chỉ dựa trên tập học[cite: 37]
print("Do sau   tap hoc   kiem dinh cheo")
diem = {}
for d in range(1, 16):
  mh = DecisionTreeClassifier(max_depth=d, random_state=0)
  diem[d] = cross_val_score(mh, X_tr, y_tr, cv=5).mean()
  hoc = accuracy_score(y_tr, mh.fit(X_tr, y_tr).predict(X_tr))
  print(f" {d:2d}     {hoc:.4f}      {diem[d]:.4f}")

d_tot = max(diem, key=diem.get)
print()

# Chọn xong mới mở tập kiểm tra, và chỉ mở một lần[cite: 37]
tot = DecisionTreeClassifier(max_depth=d_tot, random_state=0)
tot.fit(X_tr, y_tr)
print(f"Do sau tot nhat = {d_tot}, cay co {tot.get_n_leaves()} la")
acc = accuracy_score(y_te, tot.predict(X_te))
print(f"Accuracy tren tap kiem tra = {acc:.4f}")

Cay moc tu do
 do sau 13, so la 92
 tap hoc 1.0000, tap kiem tra 0.7667

Do sau   tap hoc   kiem dinh cheo
  1     0.7467      0.7244
  2     0.7711      0.7378
  3     0.8000      0.7511
  4     0.8644      0.8089
  5     0.8978      0.8333
  6     0.9133      0.8156
  7     0.9400      0.8067
  8     0.9600      0.7733
  9     0.9800      0.7800
 10     0.9911      0.7822
 11     0.9933      0.7756
 12     0.9978      0.7778
 13     1.0000      0.7733
 14     1.0000      0.7733
 15     1.0000      0.7733

Do sau tot nhat = 5, cay co 27 la
Accuracy tren tap kiem tra = 0.8533


In [7]:
# -*- coding: utf-8 -*-
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("data/khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)


def cay_sau_5():
  return DecisionTreeClassifier(max_depth=5, random_state=0)


# Cây 1: dữ liệu gốc
goc = cay_sau_5().fit(X_tr, y_tr)
du_goc = goc.predict(X_te)

# Cây 2: dữ liệu đã chuẩn hoá, trung bình tính trên riêng tập học
sc = StandardScaler().fit(X_tr)
chuan = cay_sau_5().fit(sc.transform(X_tr), y_tr)
du_chuan = chuan.predict(sc.transform(X_te))

# Cây 3: đổi thu nhập từ triệu đồng sang nghìn đồng
X_tr_n, X_te_n = X_tr.copy(), X_te.copy()
X_tr_n["thu_nhap"] = X_tr_n["thu_nhap"] * 1000
X_te_n["thu_nhap"] = X_te_n["thu_nhap"] * 1000
nghin = cay_sau_5().fit(X_tr_n, y_tr)
du_nghin = nghin.predict(X_te_n)

print("Accuracy tren tap kiem tra")
print(f"  du lieu goc         : {accuracy_score(y_te, du_goc):.4f}")
print(f"  da chuan hoa        : {accuracy_score(y_te, du_chuan):.4f}")
print(f"  thu nhap nghin dong : {accuracy_score(y_te, du_nghin):.4f}")
print()

print("So khach bi doan khac cay goc")
print(f"  da chuan hoa        : {(du_chuan != du_goc).sum()}")
print(f"  thu nhap nghin dong : {(du_nghin != du_goc).sum()}")
print()

print("Nguong cua cau hoi goc")
print(f"  du lieu goc         : {goc.tree_.threshold[0]:.2f}")
print(f"  thu nhap nghin dong : {nghin.tree_.threshold[0]:.2f}")

Accuracy tren tap kiem tra
  du lieu goc         : 0.8533
  da chuan hoa        : 0.8533
  thu nhap nghin dong : 0.8533

So khach bi doan khac cay goc
  da chuan hoa        : 0
  thu nhap nghin dong : 0

Nguong cua cau hoi goc
  du lieu goc         : 15.05
  thu nhap nghin dong : 15050.00


In [8]:
# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("data/khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Bốc mẫu có hoàn lại: rút 450 lần từ 450 khách
rng = np.random.default_rng(0)
n = len(X_tr)
chi_so = rng.integers(0, n, n)
khac = len(np.unique(chi_so))
print(f"Boc {n} lan, co {khac} khách khác nhau ({khac / n:.0%})")
print()

rung = RandomForestClassifier(n_estimators=200, random_state=0)
rung.fit(X_tr, y_tr)
print("Rung 200 cay")
print(f"  tap hoc     : {accuracy_score(y_tr, rung.predict(X_tr)):.4f}")
print(f"  tap kiem tra: {accuracy_score(y_te, rung.predict(X_te)):.4f}")
print()

print("So cay    accuracy tap kiem tra")
for m in [1, 10, 50, 200, 400]:
  r = RandomForestClassifier(n_estimators=m, random_state=0)
  r.fit(X_tr, y_tr)
  print(f" {m:4d}      {accuracy_score(y_te, r.predict(X_te)):.4f}")
print()

# Bảng tổng kết ba mô hình trên cùng tập kiểm tra
ba_mo_hinh = {
    "Cay moc tu do": DecisionTreeClassifier(random_state=0),
    "Cay sau 5": DecisionTreeClassifier(max_depth=5, random_state=0),
    "Rung 200 cay": rung,
}
print("Mo hinh          tap hoc   tap kiem tra")
for ten, mh in ba_mo_hinh.items():
  mh.fit(X_tr, y_tr)
  a = accuracy_score(y_tr, mh.predict(X_tr))
  b = accuracy_score(y_te, mh.predict(X_te))
  print(f" {ten:14s}    {a:.4f}      {b:.4f}")

Boc 450 lan, co 286 khách khác nhau (64%)

Rung 200 cay
  tap hoc     : 1.0000
  tap kiem tra: 0.9000

So cay    accuracy tap kiem tra
    1      0.7933
   10      0.8467
   50      0.8733
  200      0.9000
  400      0.9000

Mo hinh          tap hoc   tap kiem tra
 Cay moc tu do     1.0000      0.7667
 Cay sau 5         0.8978      0.8533
 Rung 200 cay      1.0000      0.9000


In [9]:
# -*- coding: utf-8 -*-
"""Bước 6: đọc độ quan trọng của đặc trưng, và thấy giới hạn của nó."""

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

rung = RandomForestClassifier(n_estimators=200, random_state=0)
rung.fit(X_tr, y_tr)
qt = pd.Series(rung.feature_importances_, index=X.columns)
print("Do quan trong, bon cot that")
print(qt.sort_values(ascending=False).round(3).to_string())
print("Tong:", round(qt.sum(), 3))
print()

# Thêm một cột toàn số ngẫu nhiên, chẳng liên quan gì tới việc trả nợ
rng = np.random.default_rng(0)
X_tr_nn = X_tr.copy()
X_tr_nn["so_ngau_nhien"] = rng.random(len(X_tr))
rung_nn = RandomForestClassifier(n_estimators=200, random_state=0)
rung_nn.fit(X_tr_nn, y_tr)
qt_nn = pd.Series(rung_nn.feature_importances_, index=X_tr_nn.columns)
print("Do quan trong, them mot cot ngau nhien")
print(qt_nn.sort_values(ascending=False).round(3).to_string())

Do quan trong, bon cot that
thu_nhap              0.479
ty_le_no              0.339
so_nguoi_phu_thuoc    0.093
co_viec_on_dinh       0.088
Tong: 1.0

Do quan trong, them mot cot ngau nhien
thu_nhap              0.427
ty_le_no              0.270
so_ngau_nhien         0.150
co_viec_on_dinh       0.077
so_nguoi_phu_thuoc    0.076


In [10]:
# -*- coding: utf-8 -*-
"""Bước 7: cây quyết định cho bài toán hồi quy, trên dữ liệu giá nhà."""

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text

df = pd.read_csv("data/gia_nha.csv")

X = df[["dien_tich"]]
y = df["gia"]

# Cùng cách chia với bài 1 để so sánh công bằng
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

cay = DecisionTreeRegressor(max_depth=3, random_state=0)
cay.fit(X_tr, y_tr)
duong = LinearRegression().fit(X_tr, y_tr)

print(f"Cay hoi quy sau 3 co {cay.get_n_leaves()} la")
print(export_text(cay, feature_names=["dien_tich"], decimals=1))

print("MAE tren tap kiem tra (ty dong)")
mae_cay = mean_absolute_error(y_te, cay.predict(X_te))
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))
print(f"  Cay sau 3   : {mae_cay:.4f}")
print(f"  Duong thang : {mae_duong:.4f}")
print()

can_moi = pd.DataFrame({"dien_tich": [60.0, 100.0, 150.0, 250.0]})
print("Du doan cho bon can moi (ty dong)")
print("  dien tich     cay    duong thang")
for dt, a, b in zip(
    can_moi["dien_tich"], cay.predict(can_moi), duong.predict(can_moi)
):
  print(f"   {dt:6.0f} m2   {a:6.2f}     {b:6.2f}")

Cay hoi quy sau 3 co 8 la
|--- dien_tich <= 73.9
|   |--- dien_tich <= 64.7
|   |   |--- dien_tich <= 50.5
|   |   |   |--- value: [4.2]
|   |   |--- dien_tich >  50.5
|   |   |   |--- value: [4.9]
|   |--- dien_tich >  64.7
|   |   |--- dien_tich <= 68.5
|   |   |   |--- value: [5.3]
|   |   |--- dien_tich >  68.5
|   |   |   |--- value: [5.9]
|--- dien_tich >  73.9
|   |--- dien_tich <= 93.8
|   |   |--- dien_tich <= 82.5
|   |   |   |--- value: [6.8]
|   |   |--- dien_tich >  82.5
|   |   |   |--- value: [7.4]
|   |--- dien_tich >  93.8
|   |   |--- dien_tich <= 107.7
|   |   |   |--- value: [8.3]
|   |   |--- dien_tich >  107.7
|   |   |   |--- value: [9.4]

MAE tren tap kiem tra (ty dong)
  Cay sau 3   : 0.5039
  Duong thang : 0.2578

Du doan cho bon can moi (ty dong)
  dien tich     cay    duong thang
       60 m2     4.92       5.12
      100 m2     8.32       8.24
      150 m2     9.40      12.13
      250 m2     9.40      19.91


Bài Tập

Bài Tập 1

In [11]:
# -*- coding: utf-8 -*-
"""Bài tập 1: Tự vẽ cây quyết định."""
import os
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Đảm bảo thư mục lưu ảnh tồn tại
os.makedirs("baitap05", exist_ok=True)

# 1. Tải và chuẩn bị dữ liệu
df = pd.read_csv("data/khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# 2. Vẽ và lưu cây 5 lá (Mục 3)
cay_5_la = DecisionTreeClassifier(max_leaf_nodes=5, random_state=0)
cay_5_la.fit(X_tr, y_tr)

plt.figure(figsize=(12, 8))
plot_tree(
    cay_5_la,
    feature_names=list(X.columns),
    class_names=["tre_han", "dung_han"],
    filled=True,
    rounded=True,
)
plt.title("Cây quyết định 5 lá")
plt.savefig("baitap05/cay_nam_la.png", bbox_inches="tight")
plt.close()
print("Đã lưu hình cay_nam_la.png")

# 3. Vẽ cây sâu 2
cay_sau_2 = DecisionTreeClassifier(max_depth=2, random_state=0)
cay_sau_2.fit(X_tr, y_tr)

plt.figure(figsize=(10, 6))
plot_tree(
    cay_sau_2,
    feature_names=list(X.columns),
    class_names=["tre_han", "dung_han"],
    filled=True,
    rounded=True,
)
plt.title("Cây quyết định độ sâu 2")
plt.savefig("baitap05/cay_sau_2.png", bbox_inches="tight")
plt.close()
print("Đã lưu hình cay_sau_2.png")

Đã lưu hình cay_nam_la.png
Đã lưu hình cay_sau_2.png


Bài tập 2

In [12]:
# -*- coding: utf-8 -*-
"""Bài tập 2: Tự tìm câu hỏi gốc trên dữ liệu thật."""
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


def gini(y_sub):
    """Tính chỉ số Gini cho một tập nhãn."""
    if len(y_sub) == 0:
        return 0.0
    p1 = y_sub.mean()
    return 1.0 - p1**2 - (1.0 - p1) ** 2


def tim_cau_hoi(X, y):
    """Thử mọi cột và mọi ngưỡng để tìm câu hỏi làm Gini giảm nhiều nhất."""
    g_goc = gini(y)
    n = len(y)

    tot_nhat_cot = None
    tot_nhat_nguong = None
    max_gini_giam = -1.0

    # Duyệt qua từng đặc trưng (cột)
    for col in X.columns:
        values = X[col].to_numpy()
        unique_vals = np.sort(np.unique(values))

        # Nếu biến định lượng có nhiều giá trị, thử điểm trung vị giữa 2 giá trị liên tiếp
        if len(unique_vals) > 2:
            nguongs = (unique_vals[:-1] + unique_vals[1:]) / 2.0
        else:
            # Nếu biến nhị phân (0 và 1), thử ngưỡng 0.5
            nguongs = [0.5]

        for nguong in nguongs:
            di_trai = values <= nguong
            y_trai = y[di_trai]
            y_phai = y[~di_trai]

            # Gini trung bình có trọng số sau khi chia
            g_sau = (len(y_trai) / n) * gini(y_trai) + (
                len(y_phai) / n
            ) * gini(y_phai)
            g_giam = g_goc - g_sau

            if g_giam > max_gini_giam:
                max_gini_giam = g_giam
                tot_nhat_cot = col
                tot_nhat_nguong = nguong

    return tot_nhat_cot, tot_nhat_nguong, max_gini_giam


# 1. Đọc dữ liệu và chia tập train/test theo đúng tỷ lệ
df = pd.read_csv("data/khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# 2. Chạy tìm câu hỏi gốc trên 450 khách hàng tập học
cot, nguong, giam = tim_cau_hoi(X_tr, y_tr.to_numpy())

print(f"Câu hỏi gốc tìm được bằng tự tính NumPy:")
print(f" - Cột            : {cot}")
print(f" - Ngưỡng         : {nguong:.3f}")
print(f" - Mức Gini giảm  : {giam:.4f}")

Câu hỏi gốc tìm được bằng tự tính NumPy:
 - Cột            : thu_nhap
 - Ngưỡng         : 15.050
 - Mức Gini giảm  : 0.1221


Bài tập 3

In [13]:
# -*- coding: utf-8 -*-
"""Bài tập 3: So sánh chỉ số Gini và Entropy."""
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

# 1. Đọc và chuẩn bị dữ liệu
df = pd.read_csv("data/khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# 2. Tìm độ sâu tốt nhất với criterion='entropy'
diem_entropy = {}
for d in range(1, 16):
    mh = DecisionTreeClassifier(
        max_depth=d, criterion="entropy", random_state=0
    )
    diem_entropy[d] = cross_val_score(mh, X_tr, y_tr, cv=5).mean()

d_tot_entropy = max(diem_entropy, key=diem_entropy.get)

# 3. Đánh giá mô hình tốt nhất trên tập kiểm tra
tot_entropy = DecisionTreeClassifier(
    max_depth=d_tot_entropy, criterion="entropy", random_state=0
)
tot_entropy.fit(X_tr, y_tr)
acc_entropy = accuracy_score(y_te, tot_entropy.predict(X_te))

print(f"--- THƯỚC ĐO ENTROPY ---")
print(
    f"Độ sâu tốt nhất (Cross-Validation): {d_tot_entropy}, CV Score: {diem_entropy[d_tot_entropy]:.4f}"
)
print(f"Accuracy trên tập kiểm tra        : {acc_entropy:.4f}\n")

print(f"--- SO SÁNH VỚI GINI (MỤC 4) ---")
print(f"Gini   : Độ sâu tốt nhất = 5, Accuracy tập kiểm tra = 0.8533")
print(
    f"Entropy: Độ sâu tốt nhất = {d_tot_entropy}, Accuracy tập kiểm tra = {acc_entropy:.4f}"
)

--- THƯỚC ĐO ENTROPY ---
Độ sâu tốt nhất (Cross-Validation): 5, CV Score: 0.8222
Accuracy trên tập kiểm tra        : 0.8467

--- SO SÁNH VỚI GINI (MỤC 4) ---
Gini   : Độ sâu tốt nhất = 5, Accuracy tập kiểm tra = 0.8533
Entropy: Độ sâu tốt nhất = 5, Accuracy tập kiểm tra = 0.8467


Bài tập 4

In [14]:
# -*- coding: utf-8 -*-
"""Bài tập 4: Cắt tỉa bằng min_samples_leaf."""
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

# 1. Chuẩn bị dữ liệu
df = pd.read_csv("data/khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# 2. Thử các giá trị min_samples_leaf
danh_sach_leaf = [1, 2, 5, 10, 20, 40]
diem_leaf = {}

print("min_samples_leaf | CV Score (5-fold)")
print("-" * 35)
for leaf in danh_sach_leaf:
    mh = DecisionTreeClassifier(min_samples_leaf=leaf, random_state=0)
    score = cross_val_score(mh, X_tr, y_tr, cv=5).mean()
    diem_leaf[leaf] = score
    print(f"{leaf:16d} | {score:.4f}")

leaf_tot = max(diem_leaf, key=diem_leaf.get)

# 3. Đánh giá mô hình tối ưu trên tập kiểm tra
tot_leaf = DecisionTreeClassifier(min_samples_leaf=leaf_tot, random_state=0)
tot_leaf.fit(X_tr, y_tr)
acc_leaf = accuracy_score(y_te, tot_leaf.predict(X_te))

print(f"\nGiá trị min_samples_leaf tốt nhất: {leaf_tot}")
print(f"Accuracy tập kiểm tra (min_samples_leaf={leaf_tot}): {acc_leaf:.4f}")
print(f"Accuracy tập kiểm tra (Cây sâu 5 - Mục 4)       : 0.8533")

min_samples_leaf | CV Score (5-fold)
-----------------------------------
               1 | 0.7733
               2 | 0.7822
               5 | 0.8067
              10 | 0.8089
              20 | 0.8089
              40 | 0.7133

Giá trị min_samples_leaf tốt nhất: 10
Accuracy tập kiểm tra (min_samples_leaf=10): 0.8467
Accuracy tập kiểm tra (Cây sâu 5 - Mục 4)       : 0.8533


Bài Tập 5

In [15]:
# -*- coding: utf-8 -*-
"""Bài tập 5: Độ quan trọng đặc trưng bằng Permutation Importance."""
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

# 1. Đọc dữ liệu và chia tập học / tập kiểm tra
df = pd.read_csv("data/khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# 2. Thêm cột số ngẫu nhiên vào cả tập học và tập kiểm tra
rng = np.random.default_rng(0)
X_tr_nn = X_tr.copy()
X_te_nn = X_te.copy()

X_tr_nn["so_ngau_nhien"] = rng.random(len(X_tr))
X_te_nn["so_ngau_nhien"] = rng.random(len(X_te))

# 3. Huấn luyện Rừng ngẫu nhiên
rung_nn = RandomForestClassifier(n_estimators=200, random_state=0)
rung_nn.fit(X_tr_nn, y_tr)

# 4. Tính Permutation Importance trên TẬP KIỂM TRA
perm_imp = permutation_importance(
    rung_nn, X_te_nn, y_te, n_repeats=20, random_state=0
)

df_perm = pd.Series(
    perm_imp.importances_mean, index=X_te_nn.columns
).sort_values(ascending=False)

print("--- ĐỘ QUAN TRỌNG BẰNG PERMUTATION IMPORTANCE (TẬP KIỂM TRA) ---")
print(df_perm.round(4).to_string())

--- ĐỘ QUAN TRỌNG BẰNG PERMUTATION IMPORTANCE (TẬP KIỂM TRA) ---
thu_nhap              0.3073
ty_le_no              0.1993
co_viec_on_dinh       0.1117
so_nguoi_phu_thuoc    0.0497
so_ngau_nhien         0.0017


Bài tập 6


In [16]:
# -*- coding: utf-8 -*-
"""Bài tập 6: Rừng ngẫu nhiên cho bài toán hồi quy."""
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

# 1. Đọc dữ liệu giá nhà
df = pd.read_csv("data/gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]

# Chia dữ liệu khớp Mục 8
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Khởi tạo và huấn luyện 3 mô hình
cay = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X_tr, y_tr)
duong = LinearRegression().fit(X_tr, y_tr)
rung = RandomForestRegressor(n_estimators=200, random_state=0).fit(X_tr, y_tr)

# 3. So sánh MAE trên tập kiểm tra
mae_cay = mean_absolute_error(y_te, cay.predict(X_te))
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))
mae_rung = mean_absolute_error(y_te, rung.predict(X_te))

print("--- MAE TRÊN TẬP KIỂM TRA (TỶ ĐỒNG) ---")
print(f" - Cây sâu 3        : {mae_cay:.4f}")
print(f" - Đường thẳng      : {mae_duong:.4f}")
print(f" - Rừng 200 cây     : {mae_rung:.4f}\n")

# 4. Dự đoán giá nhà cho căn mới 250 m2
can_250 = pd.DataFrame({"dien_tich": [250.0]})
gia_cay = cay.predict(can_250)[0]
gia_duong = duong.predict(can_250)[0]
gia_rung = rung.predict(can_250)[0]

print("--- DỰ ĐOÁN GIÁ CĂN NƠI 250 m2 ---")
print(f" - Cây sâu 3        : {gia_cay:.2f} tỷ")
print(f" - Đường thẳng      : {gia_duong:.2f} tỷ")
print(f" - Rừng 200 cây     : {gia_rung:.2f} tỷ")

--- MAE TRÊN TẬP KIỂM TRA (TỶ ĐỒNG) ---
 - Cây sâu 3        : 0.5039
 - Đường thẳng      : 0.2578
 - Rừng 200 cây     : 0.4559

--- DỰ ĐOÁN GIÁ CĂN NƠI 250 m2 ---
 - Cây sâu 3        : 9.40 tỷ
 - Đường thẳng      : 19.91 tỷ
 - Rừng 200 cây     : 9.20 tỷ
